In [1]:
!pip install -q transformers torch tiktoken bertviz datasets
!pip install -q scikit-learn pandas numpy matplotlib seaborn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 157.5/157.5 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.9/14.9 MB 62.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 2.0 MB/s eta 0:00:00


In [2]:
import warnings
warnings.filterwarnings('ignore')

import torch
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

from transformers import pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12


In [3]:
# Load the cleaned Olist dataset
df = pd.read_csv('olist_dataset_cleaned.csv')

# Target variable (same as HW2)
df['is_positive_review'] = df['review_score'].map({1:0, 2:0, 3:0, 4:1, 5:1})
df = df.dropna(subset=['is_positive_review']).copy()

# Sort by time to avoid leakage in historical features
df = df.sort_values('order_purchase_timestamp')

# Feature engineering
df['same_state'] = (df['customer_state'] == df['seller_state']).astype(int)

seller_group = df.groupby('seller_id')
df['seller_score']       = seller_group['review_score'].expanding().mean().shift().reset_index(level=0, drop=True)
df['num_previous_sales'] = seller_group['review_score'].cumcount()

cust_group = df.groupby('customer_id')
df['cust_reviews']         = cust_group['review_score'].expanding().mean().shift().reset_index(level=0, drop=True)
df['num_previous_reviews'] = cust_group['review_score'].cumcount()

num_items_per_order  = df.groupby('order_id')['order_item_id'].count()
df['num_items']      = df['order_id'].map(num_items_per_order)
df['is_repeat_customer'] = (df.groupby('customer_id').cumcount() > 0).astype(int)
df['delivery_missing']   = df['delivery_days'].isnull().astype(int)

FEATURE_COLS = [
    'delivery_days', 'delivery_vs_estimated', 'price', 'freight_value',
    'product_category_name_english', 'seller_state', 'payment_type',
    'same_state', 'seller_score', 'num_previous_sales', 'cust_reviews',
    'num_previous_reviews', 'num_items', 'freight_ratio',
    'is_repeat_customer', 'delivery_missing'
]

print(f'Full dataset: {len(df):,} records')
print(f'Records with review text: {df["review_comment_message"].notna().sum():,}')

Full dataset: 117,437 records
Records with review text: 49,995


## Part 1A: Setup and Inference

Filtering to records with non-empty review text, then sampling 500 records with random_state=42 for reproducibility. Using the same target definition as HW2: 4–5 ratings: positive (1), 1–3 ratings: negative (0).

In [4]:
# Filter to records with non-empty review text
df_text = df[df['review_comment_message'].notna() & (df['review_comment_message'].str.strip() != '')].copy()

# Sample 500 records
sample = df_text.sample(n=500, random_state=42).copy()
y_true = sample['is_positive_review'].astype(int)

print(f'Sample size: {len(sample)}')
print(f'\nActual class distribution:')
print(f'  Positive (1): {y_true.sum()} ({y_true.mean():.1%})')
print(f'  Negative (0): {(1-y_true).sum()} ({1-y_true.mean():.1%})')

Sample size: 500

Actual class distribution:
  Positive (1): 330 (66.0%)
  Negative (0): 170 (34.0%)


In [6]:
# Load the nlptown multilingual sentiment model
sentiment = pipeline("sentiment-analysis",
model="nlptown/bert-base-multilingual-uncased-sentiment")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [8]:
# Run inference on 500 reviews

texts = sample['review_comment_message'].tolist()
results = sentiment(texts, batch_size=16)

# Map star ratings to binary: 4–5 stars: positive (1), 1–3: negative (0)
def map_to_binary(label):
    stars = int(label.split()[0])
    return 1 if stars >= 4 else 0

foundation_preds  = [map_to_binary(r['label']) for r in results]
foundation_scores = [r['score'] for r in results]

sample['foundation_pred']  = foundation_preds
sample['foundation_score'] = foundation_scores

print('Inference complete!')
print(f'\nFoundation model predicted class distribution:')
print(f'  Positive (1): {sum(foundation_preds)} ({sum(foundation_preds)/len(foundation_preds):.1%})')
print(f'  Negative (0): {len(foundation_preds)-sum(foundation_preds)} ({1-sum(foundation_preds)/len(foundation_preds):.1%})')

# Show a sample of predictions
print('\nSample predictions (first 10):')
print('=' * 80)
for i, (text, true, pred, conf) in enumerate(zip(
    sample['review_comment_message'].tolist()[:10],
    y_true.tolist()[:10],
    foundation_preds[:10],
    results[:10]
)):
    status = 'CORRECT' if true == pred else 'WRONG'
    print(f'  [{status:>7s}] {conf["label"]:>10s} ({conf["score"]:.2f}) | {str(text)[:60]}')

Inference complete!

Foundation model predicted class distribution:
  Positive (1): 272 (54.4%)
  Negative (0): 228 (45.6%)

Sample predictions (first 10):
  [CORRECT]    2 stars (0.42) | TAPETE MUITO FINO, E PELOS FINOS
  [CORRECT]     1 star (0.63) | Eu comprei duas unidades do mesmo produto é apenas uma delas
  [CORRECT]    5 stars (0.28) | nota 10
  [CORRECT]     1 star (0.58) | Sem comentário , não recebi o relógio Cassio .
  [CORRECT]    5 stars (0.48) | Gostei do produto. 
  [CORRECT]    5 stars (0.60) | Empresa honesta.. chegou até antes do previsto.. produto de 
  [CORRECT]    4 stars (0.55) | SATISFEITA COM O PRODUTO
  [CORRECT]    5 stars (0.31) | PRODUTO SHOW DE BOLA
  [CORRECT]    3 stars (0.30) | SOLICITEI OUTRO MODELO E COM MAIS ML
  [CORRECT]    5 stars (0.46) | Produto foi entregue corretamente e antes do prazo estabelec


## Part 1B: Performance Comparison

Running the HW2 model on the same 500 records using order and delivery features, then comparing both models against the actual review scores.

In [9]:
# Load HW2 model and run on same 500 records
model = joblib.load('model.pkl')   # adjust path if needed

X_sample   = sample[FEATURE_COLS]
hw2_preds  = model.predict(X_sample)

print(f'HW2 model predicted class distribution:')
print(f'  Positive (1): {hw2_preds.sum()} ({hw2_preds.mean():.1%})')
print(f'  Negative (0): {len(hw2_preds)-hw2_preds.sum()} ({1-hw2_preds.mean():.1%})')

HW2 model predicted class distribution:
  Positive (1): 353 (70.6%)
  Negative (0): 147 (29.4%)


In [10]:
# Performance comparison table
print('Foundation Model (review text — zero Olist training):')
print(classification_report(y_true, foundation_preds, target_names=['Negative', 'Positive']))

print('\nHW2 Model (order features — trained on Olist):')
print(classification_report(y_true, hw2_preds, target_names=['Negative', 'Positive']))

# Summary table
comparison = pd.DataFrame([
    {
        'Model': 'Foundation Model (review text)',
        'Accuracy':  round(accuracy_score(y_true, foundation_preds),  4),
        'Precision': round(precision_score(y_true, foundation_preds), 4),
        'Recall':    round(recall_score(y_true, foundation_preds),    4),
        'F1 Score':  round(f1_score(y_true, foundation_preds),        4),
    },
    {
        'Model': 'HW2 Model (order features)',
        'Accuracy':  round(accuracy_score(y_true, hw2_preds),  4),
        'Precision': round(precision_score(y_true, hw2_preds), 4),
        'Recall':    round(recall_score(y_true, hw2_preds),    4),
        'F1 Score':  round(f1_score(y_true, hw2_preds),        4),
    },
]).set_index('Model')

print('\nSummary Comparison Table — same 500 records:')
display(comparison)

Foundation Model (review text — zero Olist training):
              precision    recall  f1-score   support

    Negative       0.68      0.91      0.77       170
    Positive       0.94      0.78      0.85       330

    accuracy                           0.82       500
   macro avg       0.81      0.84      0.81       500
weighted avg       0.85      0.82      0.82       500


HW2 Model (order features — trained on Olist):
              precision    recall  f1-score   support

    Negative       0.71      0.61      0.66       170
    Positive       0.81      0.87      0.84       330

    accuracy                           0.78       500
   macro avg       0.76      0.74      0.75       500
weighted avg       0.78      0.78      0.78       500


Summary Comparison Table — same 500 records:


,Accuracy,Precision,Recall,F1 Score
Model,,,,
Foundation Model (review text),0.820,0.9412,0.7758,0.8505
HW2 Model (order features),0.782,0.8130,0.8697,0.8404


## Part 1C: Reflection

**Which model performed better, and why?**

The foundation model achieved higher overall accuracy (82.0% vs. 78.2%) and F1 score (0.85 vs. 0.84). The foundation model excels at identifying negative reviews (recall of 91% vs. 61%), which makes sense — it reads the actual review text, where dissatisfaction is usually expressed explicitly. The HW2 model relies solely on delivery and order signals, yet still achieves comparable F1 on the positive class.

**Advantages and disadvantages of the zero-training approach:**

The foundation model's biggest advantage is that it requires no labeled Olist data, no training pipeline, it can be deployed almost immediately. The main disadvantage is that it can only operate after the review
is written, and it is computationally "heavy" compared to the sklearn pipeline, and it has no access to the structured signals like delivery delays, freight costs, seller history, that can be predictive of satisfaction even before the customer has expressed anything.

**When to use each approach:**

A foundation model seems to be the right choice when labeled data is scarce, when the task is general-purpose text classification, or when time-to-deployment is the priority. A custom model like the HW2 pipeline
is preferable when structured, domain-specific features are available, when predictions must happen before the outcome event (the review), and when inference cost and latency matter.

**Could they be combined?**

In a production system, both models serve complementary roles at different stages of the customer lifecycle. The HW2 model acts as an
early-warning system: as soon as an order is placed, it flags at-risk customers based on delivery and seller signals, giving Olist time to intervene before dissatisfaction becomes a review.
The foundation model could act as an "after-the-fact" classifier: once a review is submitted, it automatically routes negative feedback to the customer service team for follow-up.

Additionally, the foundation model could enrich the HW2 feature set itself. Currently, cust_reviews captures a customer's historical satisfaction using raw star ratings.
The foundation model could derive a text-based sentiment score from past reviews, capturing nuance that purely numerical star ratings miss (a customer who writes negatively even in 4-star reviews is a different risk profile than one who writes enthusiastically) which could make the proactive model more accurate without changing its real-time prediction capability.